# Lesson 02 - 探索 Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** 是一個統一的框架，用於構建 AI 代理。它提供了一個乾淨、可組合的架構，包含四個核心構建模組：

- **Client** – 連接到 AI 模型端點並處理通信
- **Agent** – 使用指令和工具定義封裝客戶端
- **Tools** – 使用模型可以呼叫的自訂函數擴展代理功能
- **Session** – 維護多輪互動的對話歷史

在本課程中，我們將使用這些概念建立一個**旅遊訂票代理**，用於檢查目的地可用性。


## 設定


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 了解代理框架架構

Microsoft Agent 框架採用分層架構：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` 會連接到 Azure OpenAI 部署。它負責身份驗證、請求格式化和回應解析。
2. **Agent** – 透過 `provider.create_agent()` 從客戶端建立的代理，將模型存取與指令（系統提示）及工具結合。
3. **Tools** – 使用 `@tool` 裝飾的 Python 函數，代理可以呼叫這些函數來執行動作或擷取資料。
4. **Session** – `AgentSession` 物件（透過 `agent.create_session()` 建立），用來儲存對話歷史，支持多輪對話，使代理可記住先前上下文。

讓我們逐層建立。


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 使用 @tool 裝飾器新增工具

工具讓代理人能夠進行生成文字以外的操作。`@tool` 裝飾器會將一般的 Python 函式轉換成代理人可以調用的功能。

重點：
- 使用 `Annotated[type, "description"]` 讓模型理解每個參數。
- 文件字串會成為模型看到的工具描述。
- `approval_mode="never_require"` 表示工具會自動執行，無需使用者確認。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 使用工具建立代理人

現在我們將客戶端、指令與工具結合成一個代理人。`instructions` 作為系統提示 — 它們定義了代理人的角色和行為。


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 使用會話進行多輪對話

`AgentSession`（透過 `agent.create_session()` 建立）會記錄對話中的所有訊息。透過每次呼叫 `agent.run()` 時傳遞相同的會話，代理人能夠存取完整的對話歷史，並可回顧早前的訊息。

我們傳入 `tools=[check_destination_availability]`，讓代理人在每輪對話中都能呼叫我們的可用性檢查工具。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## 總結

在本課程中，你探討了 Microsoft Agent Framework 的四大支柱：

| 概念 | 你所學到的內容 |
|---------|------------------|
| **客戶端** | `AzureAIProjectAgentProvider` 使用基於憑證的認證連接到 Azure OpenAI |
| **代理** | `provider.create_agent()` 將模型連接與指令及名稱打包 |
| **工具** | `@tool` 裝飾器將 Python 函數暴露給代理調用 |
| **會話** | `agent.create_session()` 維持多輪對話的歷史記錄 |

這些組成部分合起來創造出能進行自然對話、調用外部函數並維持上下文的代理——這是後續課程中更進階代理模式的基礎。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們致力於確保準確性，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
